# PSX Due-Diligence Agent — Qwen (DashScope)

Router graph: **init_node (router)** classifies the query, then fans out to **compliance** and/or **finance** (in parallel), and **synthesize** merges multi-agent answers into one reply.

Run cells top to bottom. To start a **fresh conversation**, re-run the *thread id* then *context* cells. To ask a new question, edit the *user input* cell and run *invoke*. Ensure `.env` has `DASHSCOPE_API_KEY`; select the `.venv` kernel.

In [1]:
import os
import uuid
from dotenv import load_dotenv

load_dotenv(override=True)  # reads .env; override=True so .env wins over any stale system env vars
os.environ.setdefault("DB_PATH", os.path.join(os.getcwd(), "data", "psx.db"))
# Finance agent reads markdown summaries from FINANCE_DATA_DIR (set it in .env once you
# move the data in; for now default to the existing finance project folder).
os.environ.setdefault("FINANCE_DATA_DIR", r"C:\Users\Pari\Documents\finance\data_md")

from app.common.utils import _get_model
from app.common.context import SessionContext
from app.graph.compile import get_compiled_agent

assert os.getenv("DASHSCOPE_API_KEY"), "Set DASHSCOPE_API_KEY in .env (see .env.example)"
print("DB_PATH          =", os.environ["DB_PATH"])
print("FINANCE_DATA_DIR =", os.environ["FINANCE_DATA_DIR"])
print("model            =", os.getenv("QWEN_MODEL", "qwen-plus"))

DB_PATH          = data/psx.db
FINANCE_DATA_DIR = C:\Users\Pari\Documents\finance\data_md
model            = qwen3.7-plus


In [2]:
# --- LangSmith tracing (EU region) ---
# Your key is an EU-region key, so use the EU endpoint. Add LANGSMITH_API_KEY to .env.
# Use the canonical LANGSMITH_* names only (do NOT also set LANGCHAIN_ENDPOINT -- mixing
# the two causes endpoint conflicts). Run this BEFORE the first invoke in a fresh kernel.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "devpost-hackathon"

assert os.getenv("LANGSMITH_API_KEY"), "Add LANGSMITH_API_KEY (EU region) to .env"
print("LangSmith tracing ON -> project:", os.environ["LANGSMITH_PROJECT"], "| endpoint: EU")

LangSmith tracing ON -> project: devpost-hackathon | endpoint: EU


In [3]:
# Compile the full PSX graph once (router -> compliance/finance -> synthesize).
agent = await get_compiled_agent("psx_agent")

In [4]:
# Thread id -- re-run this cell to start a brand-new conversation (fresh memory).
thread_id = str(uuid.uuid4())
print("thread_id =", thread_id) 

thread_id = bba2059b-0bd3-4c43-bdab-39475437f5bf


In [ ]:
# Context -- edit freely (e.g. swap model tier). Re-run after changing thread_id above.
context = SessionContext(
    thread_id=thread_id,
    model=_get_model("qwen"),
    agents=None,  
    user_id="pari",
    compliance_output="",
    finance_output="",
    synthesize_output="" 
    
    
)
config = {"configurable": {"thread_id": thread_id}}

In [6]:
# User input -- edit this, then run the invoke cell below.
user_input = "which broker has paid highest fine"

In [7]:
# Invoke the graph with the user input + context, streaming the answer.
# The graph emits chunks per source: compliance_chunk / finance_chunk (specialists working)
# and synthesize_chunk (the merged answer, only for multi-agent queries).
async for chunk in agent.astream(
    {"messages": [{"role": "user", "content": user_input}]},
    stream_mode="custom",
    context=context,
    config=config,
):  
    # print(context.agents)
    if context.agents == ["compliance_node"]:
        content = chunk.get('compliance_chunk')
    elif context.agents == ["finance_node"]:
        content = chunk.get('finance_chunk')
    else:  # multi-agent or synthesize node emitting
        content = chunk.get('synthesize_chunk')
    if content:
        print(content, end="", flush=True)
print()

Error during LLM call: Error code: 400 - {'error': {'message': "<400> InternalError.Algo.InvalidParameter: 'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_parameter_error'}, 'id': 'chatcmpl-5d177534-4566-96ce-8741-1393587f724f', 'request_id': '5d177534-4566-96ce-8741-1393587f724f'}
The broker with the highest single fine is Akhai Securities (Private) Limited, with a fine amount of PKR 5,000,000 and an order date of 4 February 2026. Please note that this action was noted as being "Under Appeal" at the time of the order.
